# ML models

Train six classifiers on four feature tables from `01_clean_dataset.ipynb`:

1. **Baseline** — IEEE-CIS columns only (no `uid`, `uid2`, or `DT_*`)
2. **Feature Engineering** — baseline plus `uid`, `uid2`, and `DT_*`
3. **Reduced Baseline** — Table 3 filters (≥90% empty, |r| > 0.98, IG < 0.001), no `uid` / `uid2` / `DT_*`
4. **Reduced Feature Engineering** — reduced table plus `uid` / `uid2` / `DT_*` if they survived the filters

Model order: logistic regression, decision tree, random forest, LightGBM, XGBoost, CatBoost.

Hyperparameters are tuned with **Optuna** (TPE, maximize ROC-AUC) on an inner temporal split of the 80% train set. The original 20% holdout is scored only after the best params are refit. Set `USE_OPTUNA = False` or `N_TRIALS = 0` to skip tuning.

Each run is appended to `saved/ml_results.parquet` (including `BestParams` and inner `TuneROC-AUC`). Charts live in `02_ml_models_result.ipynb`.

Kernel: `ai`.


In [1]:
import json
import warnings

import numpy as np
import pandas as pd
import altair as alt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable("mimetype")
%env JOBLIB_TEMP_FOLDER=/tmp


env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
from pathlib import Path
import pandas as pd

try:
    import google.colab

    IS_COLAB = True
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
SAVED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")


Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"
print(f"train {train.shape}  test {test.shape}")


train (590540, 441)  test (506691, 440)


In [4]:
import gc
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


def unwrap_estimator(model):
    if hasattr(model, "named_steps"):
        return list(model.named_steps.values())[-1]
    if hasattr(model, "estimator"):
        return model.estimator
    return model


def top_feature_importances(model, columns, n=20):
    est = unwrap_estimator(model)
    if hasattr(est, "calibrated_classifiers_"):
        inner = est.calibrated_classifiers_[0]
        est = getattr(inner, "estimator", getattr(inner, "base_estimator", est))
    if hasattr(est, "feature_importances_"):
        importances = np.asarray(est.feature_importances_, dtype=float)
    elif hasattr(est, "coef_"):
        importances = np.abs(np.asarray(est.coef_, dtype=float).ravel())
    else:
        return []
    if importances.size != len(columns):
        return []
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def positive_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


def evaluate(model, X_train, X_valid, y_train, y_valid, name):
    print(name)
    print(f"Features: {X_train.shape[1]}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    y_score = positive_scores(model, X_valid)
    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_score)
    pr_auc = average_precision_score(y_valid, y_score)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)
    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(
        classification_report(
            y_valid, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0
        )
    )
    top20 = top_feature_importances(model, X_train.columns)
    if top20:
        print("\nTop 20 feature importances:")
        for row in top20:
            print(
                f"  {row['rank']:2d}. {row['feature']:<32s} "
                f"{row['importance']:.6f}  ({row['importance_pct']:.2f}%)"
            )
    gc.collect()
    return {
        "Model": name,
        "Features": int(X_train.shape[1]),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
        "Top20Importances": json.dumps(top20),
    }


In [5]:
def make_logreg():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=1000,
                    solver="lbfgs",
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


def make_decision_tree():
    return DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_SEED, max_depth=20
    )


def make_random_forest():
    return RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
    )


def make_lightgbm():
    return LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )


def make_xgboost(y_train):
    n_neg = int((y_train == 0).sum())
    n_pos = max(int((y_train == 1).sum()), 1)
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=n_neg / n_pos,
    )


def make_catboost():
    return CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        random_seed=RANDOM_SEED,
        auto_class_weights="Balanced",
        verbose=False,
        allow_writing_files=False,
    )


# Display prefix, ModelType, builder
MODEL_SPECS = [
    ("Logistic Regression", "LogisticRegression", lambda y: make_logreg()),
    ("Decision Tree", "DecisionTree", lambda y: make_decision_tree()),
    ("RF", "RandomForest", lambda y: make_random_forest()),
    ("LightGBM", "LightGBM", lambda y: make_lightgbm()),
    ("XGBoost", "XGBoost", lambda y: make_xgboost(y)),
    ("CatBoost", "CatBoost", lambda y: make_catboost()),
]


## Optuna

TPE search maximizing **ROC-AUC** on the last `TUNE_VALID_FRAC` of the 80% train split (still before the holdout). Random Forest uses only the most recent `TUNE_MAX_ROWS` rows for inner training. The holdout is untouched until `evaluate()`.


In [6]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

USE_OPTUNA = True
N_TRIALS = 20
TUNE_VALID_FRAC = 0.2
TUNE_MAX_ROWS = {
    "RandomForest": 100_000,
}


def _scale_pos_weight(y):
    n_neg = int((y == 0).sum())
    n_pos = max(int((y == 1).sum()), 1)
    return n_neg / n_pos


def sample_params(trial, model_type):
    if model_type == "LogisticRegression":
        return {"C": trial.suggest_float("C", 1e-3, 10.0, log=True)}
    if model_type == "DecisionTree":
        return {
            "max_depth": trial.suggest_int("max_depth", 4, 32),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 64),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 40),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        }
    if model_type == "RandomForest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
            "max_depth": trial.suggest_int("max_depth", 8, 32),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        }
    if model_type == "LightGBM":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 96),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        }
    if model_type == "XGBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        }
    if model_type == "CatBoost":
        return {
            "iterations": trial.suggest_int("iterations", 200, 800, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        }
    raise ValueError(f"No search space for {model_type}")


def build_model(model_type, params, y_train, n_jobs=-1):
    p = dict(params)
    if model_type == "LogisticRegression":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="lbfgs",
                        random_state=RANDOM_SEED,
                        **p,
                    ),
                ),
            ]
        )
    if model_type == "DecisionTree":
        return DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, **p
        )
    if model_type == "RandomForest":
        return RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_SEED, n_jobs=n_jobs, **p
        )
    if model_type == "LightGBM":
        return LGBMClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            verbosity=-1,
            subsample_freq=1,
            **p,
        )
    if model_type == "XGBoost":
        return XGBClassifier(
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=_scale_pos_weight(y_train),
            **p,
        )
    if model_type == "CatBoost":
        return CatBoostClassifier(
            random_seed=RANDOM_SEED,
            auto_class_weights="Balanced",
            verbose=False,
            allow_writing_files=False,
            thread_count=n_jobs if n_jobs > 0 else -1,
            **p,
        )
    raise ValueError(f"Unknown model_type {model_type}")


def _inner_split(X, y, model_type):
    n = len(X)
    split = int(n * (1.0 - TUNE_VALID_FRAC))
    X_tr, X_va = X.iloc[:split], X.iloc[split:]
    y_tr, y_va = y.iloc[:split], y.iloc[split:]
    max_rows = TUNE_MAX_ROWS.get(model_type)
    if max_rows is not None and len(X_tr) > max_rows:
        X_tr = X_tr.iloc[-max_rows:]
        y_tr = y_tr.iloc[-max_rows:]
    return X_tr, X_va, y_tr, y_va


def _params_json(params):
    out = {}
    for key, value in params.items():
        if isinstance(value, (np.floating, np.integer)):
            value = value.item()
        out[key] = value
    return json.dumps(out)


def tune_hyperparams(model_type, X_train, y_train, n_trials=N_TRIALS):
    X_tr, X_va, y_tr, y_va = _inner_split(X_train, y_train, model_type)
    print(
        f"  Optuna {model_type}: {n_trials} trials  "
        f"inner train {len(X_tr):,}  inner valid {len(X_va):,}"
    )

    def objective(trial):
        params = sample_params(trial, model_type)
        model = build_model(model_type, params, y_tr, n_jobs=1)
        model.fit(X_tr, y_tr)
        y_score = positive_scores(model, X_va)
        auc = float(roc_auc_score(y_va, y_score))
        del model
        gc.collect()
        return auc

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    best_params = dict(study.best_params)
    print(f"  best inner ROC-AUC {study.best_value:.4f}  {best_params}")
    return best_params, float(study.best_value)


print(
    f"Optuna {optuna.__version__}  USE_OPTUNA={USE_OPTUNA}  N_TRIALS={N_TRIALS}  "
    f"inner valid frac={TUNE_VALID_FRAC}"
)


Optuna 4.9.0  USE_OPTUNA=True  N_TRIALS=20  inner valid frac=0.2


In [7]:
train = train.sort_values("TransactionDT").reset_index(drop=True)
y = train["isFraud"]
split_idx = int(len(train) * 0.8)
y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]

ID_COLS = ["isFraud", "TransactionID"]
FE_COLS = ["uid", "uid2", "DT_month", "DT_week", "DT_day", "DT_weekday", "DT_hour"]

baseline_cols = [c for c in train.columns if c not in ID_COLS + FE_COLS]
feature_cols = [c for c in train.columns if c not in ID_COLS]

X_train_baseline = train[baseline_cols].iloc[:split_idx]
X_valid_baseline = train[baseline_cols].iloc[split_idx:]
X_train_features = train[feature_cols].iloc[:split_idx]
X_valid_features = train[feature_cols].iloc[split_idx:]

experiments = [
    ("Baseline", X_train_baseline, X_valid_baseline),
    ("Feature Engineering", X_train_features, X_valid_features),
]

reduced_path = DATASET_PATH / "merged_train_reduced.parquet"
if not reduced_path.exists():
    raise FileNotFoundError(f"{reduced_path} missing — run 01_clean_dataset.ipynb")

train_reduced = pd.read_parquet(reduced_path).sort_values("TransactionDT").reset_index(drop=True)
if len(train_reduced) != len(train):
    raise ValueError(f"reduced rows {len(train_reduced):,} != full train {len(train):,}")

reduced_baseline_cols = [c for c in train_reduced.columns if c not in ID_COLS + FE_COLS]
reduced_feature_cols = [c for c in train_reduced.columns if c not in ID_COLS]
X_train_reduced_baseline = train_reduced[reduced_baseline_cols].iloc[:split_idx]
X_valid_reduced_baseline = train_reduced[reduced_baseline_cols].iloc[split_idx:]
X_train_reduced_features = train_reduced[reduced_feature_cols].iloc[:split_idx]
X_valid_reduced_features = train_reduced[reduced_feature_cols].iloc[split_idx:]

experiments.extend(
    [
        ("Reduced Baseline", X_train_reduced_baseline, X_valid_reduced_baseline),
        ("Reduced Feature Engineering", X_train_reduced_features, X_valid_reduced_features),
    ]
)

for label, X_tr, X_va in experiments:
    print(f"{label}: {X_tr.shape[1]} features  train {len(X_tr):,}  valid {len(X_va):,}")


Baseline: 432 features  train 472,432  valid 118,108
Feature Engineering: 439 features  train 472,432  valid 118,108
Reduced Baseline: 342 features  train 472,432  valid 118,108
Reduced Feature Engineering: 348 features  train 472,432  valid 118,108


In [8]:
all_results = []

RESULT_COLS = [
    "Model",
    "ModelType",
    "Dataset",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
    "TN",
    "FP",
    "FN",
    "TP",
    "TuneROC-AUC",
    "BestParams",
    "Top20Importances",
]
CORE_SUFFIXES = (
    " - Baseline",
    " - Feature Engineering",
    " - Reduced Baseline",
    " - Reduced Feature Engineering",
)


def _is_core_run(name) -> bool:
    text = str(name)
    return any(text.endswith(suffix) for suffix in CORE_SUFFIXES)


def save_results():
    if not all_results:
        return
    new_df = pd.DataFrame(all_results).drop_duplicates(subset=["Model"], keep="last")
    if "Dataset" not in new_df.columns:
        new_df["Dataset"] = new_df["Model"].astype(str).str.split(" - ", n=1).str[1]
    new_df = new_df[[c for c in RESULT_COLS if c in new_df.columns]]
    path = SAVED_PATH / "ml_results.parquet"
    if path.exists():
        old = pd.read_parquet(path)
        if "Model" in old.columns:
            old = old[old["Model"].map(_is_core_run)]
            old = old[~old["Model"].astype(str).str.startswith(("SVM -", "KNN -"))]
            if "ModelType" in old.columns:
                old = old[~old["ModelType"].isin(["SVM", "KNN"])]
            old = old[~old["Model"].isin(new_df["Model"])]
            if "Dataset" not in old.columns:
                old["Dataset"] = old["Model"].astype(str).str.split(" - ", n=1).str[1]
            old = old[[c for c in RESULT_COLS if c in old.columns]]
        new_df = pd.concat([old, new_df], ignore_index=True, sort=False)
    new_df = new_df.drop_duplicates(subset=["Model"], keep="last")
    new_df.to_parquet(path, index=False)
    print(f"saved {path}  n={len(new_df)}")


def run_family(prefix, model_type, factory):
    """Tune (optional), fit on all four datasets, and save after each."""
    for label, X_tr, X_va in experiments:
        name = f"{prefix} - {label}"
        best_params = {}
        tune_auc = None
        if USE_OPTUNA and N_TRIALS > 0:
            best_params, tune_auc = tune_hyperparams(model_type, X_tr, y_train)
            model = build_model(model_type, best_params, y_train)
        else:
            model = factory(y_train)
        row = evaluate(model, X_tr, X_va, y_train, y_valid, name)
        row["ModelType"] = model_type
        row["Dataset"] = label
        row["BestParams"] = _params_json(best_params)
        row["TuneROC-AUC"] = tune_auc
        all_results.append(row)
        save_results()
        gc.collect()

## Fit

One cell per model. Each cell tunes that model on the **four** datasets (`N_TRIALS` Optuna trials, inner temporal split), refits the best params on the full train split, scores the holdout, and writes `ml_results.parquet`. Run them in order (or skip a cell if that family is already saved). Set `USE_OPTUNA = False` to use the default factories only.


### Logistic Regression


In [9]:
run_family("Logistic Regression", "LogisticRegression", lambda y: make_logreg())

  Optuna LogisticRegression: 20 trials  inner train 377,945  inner valid 94,487


Best trial: 16. Best value: 0.853837: 100%|██████████| 20/20 [1:15:37<00:00, 226.85s/it]


  best inner ROC-AUC 0.8538  {'C': 1.9942666309136752}
Logistic Regression - Baseline
Features: 432
Accuracy           : 0.7058
Precision          : 0.0878
Recall             : 0.8039
F1 Score           : 0.1583
ROC-AUC            : 0.8335
PR-AUC             : 0.2067
Balanced Accuracy  : 0.7531
MCC                : 0.1986

Confusion Matrix:
[[80098 33946]
 [  797  3267]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9901    0.7023    0.8218    114044
       Fraud     0.0878    0.8039    0.1583      4064

    accuracy                         0.7058    118108
   macro avg     0.5390    0.7531    0.4900    118108
weighted avg     0.9591    0.7058    0.7989    118108


Top 20 feature importances:
   1. C11                              9.434433  (1.79%)
   2. V283                             8.448727  (1.60%)
   3. C14                              8.126130  (1.54%)
   4. V300                             7.689251  (1.46%)
   5. V218        

Best trial: 11. Best value: 0.8517: 100%|██████████| 20/20 [1:22:53<00:00, 248.68s/it]


  best inner ROC-AUC 0.8517  {'C': 6.051195987836167}
Logistic Regression - Feature Engineering
Features: 439
Accuracy           : 0.7091
Precision          : 0.0885
Recall             : 0.8022
F1 Score           : 0.1595
ROC-AUC            : 0.8334
PR-AUC             : 0.2094
Balanced Accuracy  : 0.7540
MCC                : 0.1999

Confusion Matrix:
[[80488 33556]
 [  804  3260]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9901    0.7058    0.8241    114044
       Fraud     0.0885    0.8022    0.1595      4064

    accuracy                         0.7091    118108
   macro avg     0.5393    0.7540    0.4918    118108
weighted avg     0.9591    0.7091    0.8012    118108


Top 20 feature importances:
   1. C11                              12.089177  (1.59%)
   2. V218                             10.743006  (1.41%)
   3. V300                             10.678061  (1.41%)
   4. V258                             10.297280  (1.36%)
   5

Best trial: 14. Best value: 0.848575: 100%|██████████| 20/20 [1:09:56<00:00, 209.83s/it]


  best inner ROC-AUC 0.8486  {'C': 9.177237615941623}
Logistic Regression - Reduced Baseline
Features: 342
Accuracy           : 0.6881
Precision          : 0.0834
Recall             : 0.8076
F1 Score           : 0.1513
ROC-AUC            : 0.8248
PR-AUC             : 0.1771
Balanced Accuracy  : 0.7457
MCC                : 0.1901

Confusion Matrix:
[[77993 36051]
 [  782  3282]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9901    0.6839    0.8090    114044
       Fraud     0.0834    0.8076    0.1513      4064

    accuracy                         0.6881    118108
   macro avg     0.5368    0.7457    0.4801    118108
weighted avg     0.9589    0.6881    0.7863    118108


Top 20 feature importances:
   1. V218                             14.410779  (1.81%)
   2. V45                              14.051587  (1.77%)
   3. V258                             13.816448  (1.74%)
   4. V87                              12.568217  (1.58%)
   5. V

Best trial: 7. Best value: 0.846189: 100%|██████████| 20/20 [1:09:04<00:00, 207.22s/it]


  best inner ROC-AUC 0.8462  {'C': 2.9154431891537547}
Logistic Regression - Reduced Feature Engineering
Features: 348
Accuracy           : 0.6976
Precision          : 0.0850
Recall             : 0.7975
F1 Score           : 0.1536
ROC-AUC            : 0.8248
PR-AUC             : 0.1802
Balanced Accuracy  : 0.7457
MCC                : 0.1916

Confusion Matrix:
[[79147 34897]
 [  823  3241]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9897    0.6940    0.8159    114044
       Fraud     0.0850    0.7975    0.1536      4064

    accuracy                         0.6976    118108
   macro avg     0.5373    0.7457    0.4847    118108
weighted avg     0.9586    0.6976    0.7931    118108


Top 20 feature importances:
   1. V218                             13.074718  (1.81%)
   2. V45                              12.466767  (1.73%)
   3. V258                             12.189907  (1.69%)
   4. V87                              11.055286  (1.

### Decision Tree


In [10]:
run_family("Decision Tree", "DecisionTree", lambda y: make_decision_tree())

  Optuna DecisionTree: 20 trials  inner train 377,945  inner valid 94,487


Best trial: 18. Best value: 0.838522: 100%|██████████| 20/20 [03:00<00:00,  9.05s/it]


  best inner ROC-AUC 0.8385  {'max_depth': 8, 'min_samples_leaf': 43, 'min_samples_split': 20, 'max_features': 'sqrt'}
Decision Tree - Baseline
Features: 432
Accuracy           : 0.8150
Precision          : 0.1167
Recall             : 0.6663
F1 Score           : 0.1986
ROC-AUC            : 0.8251
PR-AUC             : 0.3310
Balanced Accuracy  : 0.7433
MCC                : 0.2233

Confusion Matrix:
[[93551 20493]
 [ 1356  2708]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9857    0.8203    0.8954    114044
       Fraud     0.1167    0.6663    0.1986      4064

    accuracy                         0.8150    118108
   macro avg     0.5512    0.7433    0.5470    118108
weighted avg     0.9558    0.8150    0.8715    118108


Top 20 feature importances:
   1. V218                             0.330275  (33.03%)
   2. V102                             0.135950  (13.59%)
   3. C8                               0.108614  (10.86%)
   4. card6   

Best trial: 18. Best value: 0.845103: 100%|██████████| 20/20 [03:04<00:00,  9.24s/it]


  best inner ROC-AUC 0.8451  {'max_depth': 13, 'min_samples_leaf': 49, 'min_samples_split': 15, 'max_features': 'sqrt'}
Decision Tree - Feature Engineering
Features: 439
Accuracy           : 0.8181
Precision          : 0.1198
Recall             : 0.6754
F1 Score           : 0.2035
ROC-AUC            : 0.7965
PR-AUC             : 0.3368
Balanced Accuracy  : 0.7493
MCC                : 0.2298

Confusion Matrix:
[[93874 20170]
 [ 1319  2745]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9861    0.8231    0.8973    114044
       Fraud     0.1198    0.6754    0.2035      4064

    accuracy                         0.8181    118108
   macro avg     0.5530    0.7493    0.5504    118108
weighted avg     0.9563    0.8181    0.8734    118108


Top 20 feature importances:
   1. V243                             0.219659  (21.97%)
   2. V90                              0.175594  (17.56%)
   3. C13                              0.060702  (6.07%)
   

Best trial: 12. Best value: 0.836163: 100%|██████████| 20/20 [02:29<00:00,  7.47s/it]


  best inner ROC-AUC 0.8362  {'max_depth': 9, 'min_samples_leaf': 64, 'min_samples_split': 32, 'max_features': 'sqrt'}
Decision Tree - Reduced Baseline
Features: 342
Accuracy           : 0.8237
Precision          : 0.1210
Recall             : 0.6580
F1 Score           : 0.2044
ROC-AUC            : 0.8221
PR-AUC             : 0.3320
Balanced Accuracy  : 0.7438
MCC                : 0.2279

Confusion Matrix:
[[94616 19428]
 [ 1390  2674]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9855    0.8296    0.9009    114044
       Fraud     0.1210    0.6580    0.2044      4064

    accuracy                         0.8237    118108
   macro avg     0.5533    0.7438    0.5526    118108
weighted avg     0.9558    0.8237    0.8769    118108


Top 20 feature importances:
   1. V70                              0.301747  (30.17%)
   2. V275                             0.180378  (18.04%)
   3. card3                            0.070988  (7.10%)
   4. C

Best trial: 17. Best value: 0.848244: 100%|██████████| 20/20 [04:05<00:00, 12.26s/it]


  best inner ROC-AUC 0.8482  {'max_depth': 8, 'min_samples_leaf': 26, 'min_samples_split': 19, 'max_features': None}
Decision Tree - Reduced Feature Engineering
Features: 348
Accuracy           : 0.8306
Precision          : 0.1303
Recall             : 0.6912
F1 Score           : 0.2193
ROC-AUC            : 0.8307
PR-AUC             : 0.3506
Balanced Accuracy  : 0.7634
MCC                : 0.2486

Confusion Matrix:
[[95294 18750]
 [ 1255  2809]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9870    0.8356    0.9050    114044
       Fraud     0.1303    0.6912    0.2193      4064

    accuracy                         0.8306    118108
   macro avg     0.5586    0.7634    0.5621    118108
weighted avg     0.9575    0.8306    0.8814    118108


Top 20 feature importances:
   1. V258                             0.287585  (28.76%)
   2. V317                             0.172157  (17.22%)
   3. C14                              0.129141  (12.91

### Random Forest


In [11]:
run_family("RF", "RandomForest", lambda y: make_random_forest())

  Optuna RandomForest: 20 trials  inner train 100,000  inner valid 94,487


Best trial: 14. Best value: 0.900845: 100%|██████████| 20/20 [11:49<00:00, 35.45s/it]


  best inner ROC-AUC 0.9008  {'n_estimators': 250, 'max_depth': 25, 'min_samples_leaf': 13, 'max_features': 'sqrt'}
RF - Baseline
Features: 432
Accuracy           : 0.9295
Precision          : 0.2690
Recall             : 0.6097
F1 Score           : 0.3733
ROC-AUC            : 0.8959
PR-AUC             : 0.4875
Balanced Accuracy  : 0.7753
MCC                : 0.3743

Confusion Matrix:
[[107309   6735]
 [  1586   2478]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9854    0.9409    0.9627    114044
       Fraud     0.2690    0.6097    0.3733      4064

    accuracy                         0.9295    118108
   macro avg     0.6272    0.7753    0.6680    118108
weighted avg     0.9608    0.9295    0.9424    118108

saved d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet  n=24
  Optuna RandomForest: 20 trials  inner train 100,000  inner valid 94,487


Best trial: 14. Best value: 0.90011: 100%|██████████| 20/20 [11:41<00:00, 35.09s/it] 


  best inner ROC-AUC 0.9001  {'n_estimators': 250, 'max_depth': 25, 'min_samples_leaf': 13, 'max_features': 'sqrt'}
RF - Feature Engineering
Features: 439
Accuracy           : 0.9305
Precision          : 0.2710
Recall             : 0.6031
F1 Score           : 0.3740
ROC-AUC            : 0.8961
PR-AUC             : 0.4885
Balanced Accuracy  : 0.7726
MCC                : 0.3738

Confusion Matrix:
[[107452   6592]
 [  1613   2451]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9852    0.9422    0.9632    114044
       Fraud     0.2710    0.6031    0.3740      4064

    accuracy                         0.9305    118108
   macro avg     0.6281    0.7726    0.6686    118108
weighted avg     0.9606    0.9305    0.9429    118108

saved d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet  n=24
  Optuna RandomForest: 20 trials  inner train 100,000  inner valid 94,487


Best trial: 16. Best value: 0.898558: 100%|██████████| 20/20 [10:14<00:00, 30.74s/it]


  best inner ROC-AUC 0.8986  {'n_estimators': 250, 'max_depth': 23, 'min_samples_leaf': 8, 'max_features': 'sqrt'}
RF - Reduced Baseline
Features: 342
Accuracy           : 0.9406
Precision          : 0.3046
Recall             : 0.5669
F1 Score           : 0.3963
ROC-AUC            : 0.8907
PR-AUC             : 0.4858
Balanced Accuracy  : 0.7604
MCC                : 0.3877

Confusion Matrix:
[[108783   5261]
 [  1760   2304]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9841    0.9539    0.9687    114044
       Fraud     0.3046    0.5669    0.3963      4064

    accuracy                         0.9406    118108
   macro avg     0.6443    0.7604    0.6825    118108
weighted avg     0.9607    0.9406    0.9490    118108

saved d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet  n=24
  Optuna RandomForest: 20 trials  inner train 100,000  inner valid 94,487


Best trial: 17. Best value: 0.900672: 100%|██████████| 20/20 [12:23<00:00, 37.18s/it]


  best inner ROC-AUC 0.9007  {'n_estimators': 400, 'max_depth': 23, 'min_samples_leaf': 7, 'max_features': 'sqrt'}
RF - Reduced Feature Engineering
Features: 348
Accuracy           : 0.9455
Precision          : 0.3271
Recall             : 0.5527
F1 Score           : 0.4110
ROC-AUC            : 0.8922
PR-AUC             : 0.4894
Balanced Accuracy  : 0.7561
MCC                : 0.3990

Confusion Matrix:
[[109424   4620]
 [  1818   2246]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9837    0.9595    0.9714    114044
       Fraud     0.3271    0.5527    0.4110      4064

    accuracy                         0.9455    118108
   macro avg     0.6554    0.7561    0.6912    118108
weighted avg     0.9611    0.9455    0.9521    118108

saved d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet  n=24


### LightGBM


In [12]:
run_family("LightGBM", "LightGBM", lambda y: make_lightgbm())

  Optuna LightGBM: 20 trials  inner train 377,945  inner valid 94,487


Best trial: 19. Best value: 0.923896: 100%|██████████| 20/20 [25:34<00:00, 76.71s/it] 


  best inner ROC-AUC 0.9239  {'n_estimators': 700, 'learning_rate': 0.03702917556373974, 'num_leaves': 95, 'max_depth': 10, 'min_child_samples': 59, 'subsample': 0.8160380080454375, 'colsample_bytree': 0.6733068413073444}
LightGBM - Baseline
Features: 432
Accuracy           : 0.9421
Precision          : 0.3286
Recall             : 0.6533
F1 Score           : 0.4373
ROC-AUC            : 0.9109
PR-AUC             : 0.5498
Balanced Accuracy  : 0.8029
MCC                : 0.4374

Confusion Matrix:
[[108620   5424]
 [  1409   2655]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9872    0.9524    0.9695    114044
       Fraud     0.3286    0.6533    0.4373      4064

    accuracy                         0.9421    118108
   macro avg     0.6579    0.8029    0.7034    118108
weighted avg     0.9645    0.9421    0.9512    118108


Top 20 feature importances:
   1. card1                            3833.000000  (5.89%)
   2. TransactionDT       

Best trial: 19. Best value: 0.925101: 100%|██████████| 20/20 [26:21<00:00, 79.08s/it] 


  best inner ROC-AUC 0.9251  {'n_estimators': 700, 'learning_rate': 0.03702917556373974, 'num_leaves': 95, 'max_depth': 10, 'min_child_samples': 61, 'subsample': 0.7330736119556676, 'colsample_bytree': 0.6676709278132404}
LightGBM - Feature Engineering
Features: 439
Accuracy           : 0.9455
Precision          : 0.3467
Recall             : 0.6587
F1 Score           : 0.4543
ROC-AUC            : 0.9163
PR-AUC             : 0.5593
Balanced Accuracy  : 0.8072
MCC                : 0.4531

Confusion Matrix:
[[108999   5045]
 [  1387   2677]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9874    0.9558    0.9713    114044
       Fraud     0.3467    0.6587    0.4543      4064

    accuracy                         0.9455    118108
   macro avg     0.6671    0.8072    0.7128    118108
weighted avg     0.9654    0.9455    0.9535    118108


Top 20 feature importances:
   1. card1                            2968.000000  (4.54%)
   2. Transacti

Best trial: 12. Best value: 0.92349: 100%|██████████| 20/20 [21:57<00:00, 65.87s/it] 


  best inner ROC-AUC 0.9235  {'n_estimators': 800, 'learning_rate': 0.058392510717711, 'num_leaves': 96, 'max_depth': 12, 'min_child_samples': 54, 'subsample': 0.8072636662657275, 'colsample_bytree': 0.7235819178002387}
LightGBM - Reduced Baseline
Features: 342
Accuracy           : 0.9642
Precision          : 0.4822
Recall             : 0.5672
F1 Score           : 0.5213
ROC-AUC            : 0.9118
PR-AUC             : 0.5550
Balanced Accuracy  : 0.7727
MCC                : 0.5045

Confusion Matrix:
[[111569   2475]
 [  1759   2305]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9845    0.9783    0.9814    114044
       Fraud     0.4822    0.5672    0.5213      4064

    accuracy                         0.9642    118108
   macro avg     0.7333    0.7727    0.7513    118108
weighted avg     0.9672    0.9642    0.9655    118108


Top 20 feature importances:
   1. card1                            5426.000000  (7.15%)
   2. TransactionDT 

Best trial: 16. Best value: 0.925851: 100%|██████████| 20/20 [22:09<00:00, 66.47s/it]


  best inner ROC-AUC 0.9259  {'n_estimators': 500, 'learning_rate': 0.04003965271318858, 'num_leaves': 82, 'max_depth': 10, 'min_child_samples': 60, 'subsample': 0.7545561730804785, 'colsample_bytree': 0.7607586350559844}
LightGBM - Reduced Feature Engineering
Features: 348
Accuracy           : 0.9283
Precision          : 0.2787
Recall             : 0.6826
F1 Score           : 0.3958
ROC-AUC            : 0.9129
PR-AUC             : 0.5381
Balanced Accuracy  : 0.8098
MCC                : 0.4066

Confusion Matrix:
[[106865   7179]
 [  1290   2774]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9881    0.9371    0.9619    114044
       Fraud     0.2787    0.6826    0.3958      4064

    accuracy                         0.9283    118108
   macro avg     0.6334    0.8098    0.6788    118108
weighted avg     0.9637    0.9283    0.9424    118108


Top 20 feature importances:
   1. TransactionDT                    1850.000000  (4.58%)
   2. T

### XGBoost


In [13]:
run_family("XGBoost", "XGBoost", make_xgboost)

  Optuna XGBoost: 20 trials  inner train 377,945  inner valid 94,487


Best trial: 11. Best value: 0.922959: 100%|██████████| 20/20 [1:00:10<00:00, 180.55s/it]


  best inner ROC-AUC 0.9230  {'n_estimators': 500, 'learning_rate': 0.05156911709595948, 'max_depth': 10, 'min_child_weight': 9.905500691040231, 'subsample': 0.8708817658975764, 'colsample_bytree': 0.7930734353629822, 'gamma': 1.461192327509932}
XGBoost - Baseline
Features: 432
Accuracy           : 0.9647
Precision          : 0.4887
Recall             : 0.5706
F1 Score           : 0.5265
ROC-AUC            : 0.9114
PR-AUC             : 0.5624
Balanced Accuracy  : 0.7747
MCC                : 0.5099

Confusion Matrix:
[[111618   2426]
 [  1745   2319]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9846    0.9787    0.9817    114044
       Fraud     0.4887    0.5706    0.5265      4064

    accuracy                         0.9647    118108
   macro avg     0.7367    0.7747    0.7541    118108
weighted avg     0.9675    0.9647    0.9660    118108


Top 20 feature importances:
   1. V258                             0.136173  (13.62%)
   2.

Best trial: 14. Best value: 0.924739: 100%|██████████| 20/20 [1:04:56<00:00, 194.82s/it]


  best inner ROC-AUC 0.9247  {'n_estimators': 700, 'learning_rate': 0.07627467579795179, 'max_depth': 10, 'min_child_weight': 8.34432054049989, 'subsample': 0.8332908615865308, 'colsample_bytree': 0.8858612346238909, 'gamma': 1.9965833820037882}
XGBoost - Feature Engineering
Features: 439
Accuracy           : 0.9749
Precision          : 0.6805
Recall             : 0.5074
F1 Score           : 0.5813
ROC-AUC            : 0.9196
PR-AUC             : 0.5965
Balanced Accuracy  : 0.7494
MCC                : 0.5752

Confusion Matrix:
[[113076    968]
 [  2002   2062]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9826    0.9915    0.9870    114044
       Fraud     0.6805    0.5074    0.5813      4064

    accuracy                         0.9749    118108
   macro avg     0.8316    0.7494    0.7842    118108
weighted avg     0.9722    0.9749    0.9731    118108


Top 20 feature importances:
   1. V258                             0.210716  (21

Best trial: 11. Best value: 0.92092: 100%|██████████| 20/20 [45:27<00:00, 136.37s/it] 


  best inner ROC-AUC 0.9209  {'n_estimators': 500, 'learning_rate': 0.05156911709595948, 'max_depth': 10, 'min_child_weight': 9.905500691040231, 'subsample': 0.8708817658975764, 'colsample_bytree': 0.7930734353629822, 'gamma': 1.461192327509932}
XGBoost - Reduced Baseline
Features: 342
Accuracy           : 0.9653
Precision          : 0.4959
Recall             : 0.5605
F1 Score           : 0.5262
ROC-AUC            : 0.9120
PR-AUC             : 0.5604
Balanced Accuracy  : 0.7701
MCC                : 0.5093

Confusion Matrix:
[[111728   2316]
 [  1786   2278]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9843    0.9797    0.9820    114044
       Fraud     0.4959    0.5605    0.5262      4064

    accuracy                         0.9653    118108
   macro avg     0.7401    0.7701    0.7541    118108
weighted avg     0.9675    0.9653    0.9663    118108


Top 20 feature importances:
   1. V258                             0.159378  (15.94

Best trial: 12. Best value: 0.922654: 100%|██████████| 20/20 [49:06<00:00, 147.35s/it]


  best inner ROC-AUC 0.9227  {'n_estimators': 600, 'learning_rate': 0.05367916265026271, 'max_depth': 10, 'min_child_weight': 9.990883610300589, 'subsample': 0.8391998921110788, 'colsample_bytree': 0.8072636662657275, 'gamma': 1.4684735955100456}
XGBoost - Reduced Feature Engineering
Features: 348
Accuracy           : 0.9704
Precision          : 0.5761
Recall             : 0.5327
F1 Score           : 0.5536
ROC-AUC            : 0.9163
PR-AUC             : 0.5717
Balanced Accuracy  : 0.7594
MCC                : 0.5387

Confusion Matrix:
[[112451   1593]
 [  1899   2165]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9834    0.9860    0.9847    114044
       Fraud     0.5761    0.5327    0.5536      4064

    accuracy                         0.9704    118108
   macro avg     0.7797    0.7594    0.7691    118108
weighted avg     0.9694    0.9704    0.9699    118108


Top 20 feature importances:
   1. V258                             0.19

### CatBoost


In [14]:
run_family("CatBoost", "CatBoost", lambda y: make_catboost())

  Optuna CatBoost: 20 trials  inner train 377,945  inner valid 94,487


Best trial: 16. Best value: 0.915955: 100%|██████████| 20/20 [3:04:55<00:00, 554.79s/it]  


  best inner ROC-AUC 0.9160  {'iterations': 800, 'learning_rate': 0.053769175187915456, 'depth': 6, 'l2_leaf_reg': 3.5497480217818174}
CatBoost - Baseline
Features: 432
Accuracy           : 0.9025
Precision          : 0.2228
Recall             : 0.7362
F1 Score           : 0.3420
ROC-AUC            : 0.9078
PR-AUC             : 0.5156
Balanced Accuracy  : 0.8223
MCC                : 0.3701

Confusion Matrix:
[[103604  10440]
 [  1072   2992]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9898    0.9085    0.9474    114044
       Fraud     0.2228    0.7362    0.3420      4064

    accuracy                         0.9025    118108
   macro avg     0.6063    0.8223    0.6447    118108
weighted avg     0.9634    0.9025    0.9265    118108


Top 20 feature importances:
   1. card2                            4.971625  (4.97%)
   2. C1                               4.610614  (4.61%)
   3. C13                              4.428543  (4.43%)
  

Best trial: 13. Best value: 0.919008: 100%|██████████| 20/20 [3:05:48<00:00, 557.44s/it]  


  best inner ROC-AUC 0.9190  {'iterations': 800, 'learning_rate': 0.03283396276465817, 'depth': 7, 'l2_leaf_reg': 1.0960177507411952}
CatBoost - Feature Engineering
Features: 439
Accuracy           : 0.8987
Precision          : 0.2149
Recall             : 0.7323
F1 Score           : 0.3323
ROC-AUC            : 0.9087
PR-AUC             : 0.5065
Balanced Accuracy  : 0.8185
MCC                : 0.3609

Confusion Matrix:
[[103173  10871]
 [  1088   2976]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9896    0.9047    0.9452    114044
       Fraud     0.2149    0.7323    0.3323      4064

    accuracy                         0.8987    118108
   macro avg     0.6022    0.8185    0.6388    118108
weighted avg     0.9629    0.8987    0.9241    118108


Top 20 feature importances:
   1. C1                               5.125815  (5.13%)
   2. card2                            4.578208  (4.58%)
   3. C13                              4.273402  

Best trial: 12. Best value: 0.915534: 100%|██████████| 20/20 [1:59:48<00:00, 359.41s/it]  


  best inner ROC-AUC 0.9155  {'iterations': 700, 'learning_rate': 0.08322969653611181, 'depth': 6, 'l2_leaf_reg': 6.541633372363897}
CatBoost - Reduced Baseline
Features: 342
Accuracy           : 0.9099
Precision          : 0.2356
Recall             : 0.7205
F1 Score           : 0.3551
ROC-AUC            : 0.9093
PR-AUC             : 0.5119
Balanced Accuracy  : 0.8186
MCC                : 0.3785

Confusion Matrix:
[[104544   9500]
 [  1136   2928]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9893    0.9167    0.9516    114044
       Fraud     0.2356    0.7205    0.3551      4064

    accuracy                         0.9099    118108
   macro avg     0.6124    0.8186    0.6533    118108
weighted avg     0.9633    0.9099    0.9311    118108


Top 20 feature importances:
   1. C1                               8.669153  (8.67%)
   2. card2                            4.751526  (4.75%)
   3. card1                            4.673341  (4.6

Best trial: 17. Best value: 0.91822: 100%|██████████| 20/20 [2:28:08<00:00, 444.45s/it]   


  best inner ROC-AUC 0.9182  {'iterations': 650, 'learning_rate': 0.04955057618136131, 'depth': 7, 'l2_leaf_reg': 2.2626041404726016}
CatBoost - Reduced Feature Engineering
Features: 348
Accuracy           : 0.9034
Precision          : 0.2227
Recall             : 0.7261
F1 Score           : 0.3409
ROC-AUC            : 0.9095
PR-AUC             : 0.5094
Balanced Accuracy  : 0.8179
MCC                : 0.3673

Confusion Matrix:
[[103746  10298]
 [  1113   2951]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9894    0.9097    0.9479    114044
       Fraud     0.2227    0.7261    0.3409      4064

    accuracy                         0.9034    118108
   macro avg     0.6061    0.8179    0.6444    118108
weighted avg     0.9630    0.9034    0.9270    118108


Top 20 feature importances:
   1. C1                               8.405153  (8.41%)
   2. card2                            4.525378  (4.53%)
   3. C13                              3.